# RealThor v0.1 — Listing Supply Intelligence

**Your task in this exercise:** refactor this notebook to make RealThor reproducible.

Right now:
- The system prompt is written **inline** in this notebook.
- The data files are in the **same flat folder** as this notebook.

By the end, you should have:
- A `realthor/` project folder initialized with `uv`
- Data moved to `realthor/data/`
- The system prompt extracted to `realthor/prompts/system_prompt.md` and loaded from file
- This notebook (updated) saved as `realthor/realthor_v01.ipynb`
- A git history with at least 3 commits

## Setup

In [ ]:
import os
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv

In [ ]:
load_dotenv()

In [ ]:
provider = os.getenv("LLM_PROVIDER", "openai")
if provider == "vocareum":
    client = OpenAI(base_url="https://openai.vocareum.com/v1", api_key=os.getenv("VOCAREUM_API_KEY"))
else:
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

## Load Data

> **TODO:** After creating `realthor/data/`, update these paths to `data/demand_signals.csv` and `data/supply_snapshot.csv`.

In [ ]:
demand = pd.read_csv("demand_signals.csv")
display(demand)

In [ ]:
supply = pd.read_csv("supply_snapshot.csv")
display(supply)

## System Prompt

> **TODO:** Cut this inline string, paste it into `realthor/prompts/system_prompt.md`,
> then replace the string below with:
> ```python
> with open("prompts/system_prompt.md") as f:
>     SYSTEM_PROMPT = f.read()
> ```

In [ ]:
# NOTE: this prompt is intentionally inline — your job is to extract it
SYSTEM_PROMPT = """
You are RealThor, a Listing Supply Intelligence agent for a PropTech operating in Ciudaty.

Your task is to analyze demand and supply signals for a given region and property profile.

Use the available data to recommend whether the company should prioritize bringing more
properties of that type to the platform.

Your recommendation must include:
- priority: high, medium, or low
- reasoning based on demand
- reasoning based on supply
- recommended action
- confidence: high, medium, or low

Do not make claims that are not supported by the provided data.
"""

## RealThor Functions

In [ ]:
def build_context(region: str, property_type: str, bedrooms: int) -> str:
    d = demand[
        (demand["region"] == region)
        & (demand["property_type"] == property_type)
        & (demand["bedrooms"] == bedrooms)
    ]
    s = supply[
        (supply["region"] == region)
        & (supply["property_type"] == property_type)
        & (supply["bedrooms"] == bedrooms)
    ]

    if d.empty or s.empty:
        return "No data available for the requested segment."

    return (
        f"Region: {region}\n"
        f"Property: {bedrooms}-bedroom {property_type}\n\n"
        f"Demand signals:\n{d.to_string(index=False)}\n\n"
        f"Supply snapshot:\n{s.to_string(index=False)}"
    )

In [ ]:
def ask_realthor(region: str, property_type: str, bedrooms: int) -> str:
    context = build_context(region, property_type, bedrooms)
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": context},
        ],
    )
    return response.choices[0].message.content

## Run a Recommendation

In [ ]:
recommendation = ask_realthor(region="west", property_type="apartment", bedrooms=2)
print(recommendation)